In [4]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,roc_auc_score, confusion_matrix)

df = pd.read_csv(r"C:\Data Science\Data Set (Practice)\forage\customer_booking.csv", encoding="ISO-8859-1")
print("Shape:", df.shape, "| missing values:", int(df.isna().sum().sum()))
print("Booking rate:", round(df.booking_complete.mean(), 4))

day_map = {"Mon": 1, "Tue": 2, "Wed": 3, "Thu": 4, "Fri": 5, "Sat": 6, "Sun": 7}
df["flight_day"] = df["flight_day"].map(day_map)

# new features
df["total_extras"]   = df.wants_extra_baggage + df.wants_preferred_seat + df.wants_in_flight_meals
df["is_weekend"]     = df.flight_day.isin([6, 7]).astype(int)
df["lead_per_pax"]   = df.purchase_lead / df.num_passengers
df["stay_x_pax"]     = df.length_of_stay * df.num_passengers
df["is_night_flight"] = df.flight_hour.isin(list(range(0, 6)) + [22, 23]).astype(int)

# high-cardinality columns: keep the most common values, group the rest as "Other"
def top_n(s, n):
    keep = s.value_counts().nlargest(n).index
    return s.where(s.isin(keep), "Other")
df["booking_origin"] = top_n(df["booking_origin"], 10)
df["route"]          = top_n(df["route"], 10)

# one-hot encode the text columns
X = pd.get_dummies(df.drop(columns="booking_complete"),columns=["sales_channel", "trip_type", "booking_origin", "route"],dtype=int)
y = df["booking_complete"]
print("Features used:", X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features="sqrt",
                            class_weight="balanced_subsample", n_jobs=-1, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
res = cross_validate(rf, X_train, y_train, cv=cv, n_jobs=1,scoring=["roc_auc", "accuracy", "precision", "recall", "f1"])
print("\n5-fold cross-validation (mean +/- std)")
for m in ["roc_auc", "accuracy", "precision", "recall", "f1"]:
    v = res["test_" + m]
    print(f"  {m:10s} {v.mean():.3f} +/- {v.std():.3f}")

rf.fit(X_train, y_train)
proba = rf.predict_proba(X_test)[:, 1]
pred  = (proba >= 0.5).astype(int)
print("\nHold-out test set (20%)")
print("  AUC       ", round(roc_auc_score(y_test, proba), 3))
print("  Accuracy  ", round(accuracy_score(y_test, pred), 3))
print("  Precision ", round(precision_score(y_test, pred), 3))
print("  Recall    ", round(recall_score(y_test, pred), 3))
print("  F1        ", round(f1_score(y_test, pred), 3))
print("  Confusion matrix [[TN FP],[FN TP]]:\n", confusion_matrix(y_test, pred))

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 15 features:\n", imp.head(15).round(4))
imp.to_csv("feature_importance.csv", header=["importance"])

top = imp.head(12).sort_values()
plt.figure(figsize=(8, 5))
plt.barh(top.index, top.values, color="#1F3A93")
plt.xlabel("Importance (share of total)")
plt.title("Top 12 drivers of a customer completing a booking")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=200)

Shape: (50000, 14) | missing values: 0
Booking rate: 0.1496
Features used: 41

5-fold cross-validation (mean +/- std)
  roc_auc    0.766 +/- 0.005
  accuracy   0.783 +/- 0.004
  precision  0.344 +/- 0.009
  recall     0.497 +/- 0.013
  f1         0.407 +/- 0.010

Hold-out test set (20%)
  AUC        0.778
  Accuracy   0.787
  Precision  0.358
  Recall     0.532
  F1         0.428
  Confusion matrix [[TN FP],[FN TP]]:
 [[7074 1430]
 [ 700  796]]

Top 15 features:
 booking_origin_Australia      0.1018
purchase_lead                 0.0987
lead_per_pax                  0.0983
length_of_stay                0.0838
stay_x_pax                    0.0828
booking_origin_Malaysia       0.0748
flight_hour                   0.0739
flight_duration               0.0708
flight_day                    0.0481
total_extras                  0.0276
num_passengers                0.0211
booking_origin_Other          0.0173
wants_extra_baggage           0.0159
booking_origin_South Korea    0.0131
booking_origin